In [ ]:
import torch
print("CUDA beschikbaar:", torch.cuda.is_available())
print("GPU naam:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "geen GPU")

In [ ]:
!pip install -q transformers

In [ ]:
import pandas as pd

URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv"
df = pd.read_csv(URL)

print("Aantal rijen:", len(df))
print("Kolommen:", df.columns.tolist())
df.head()

In [ ]:
print(df["p_np"].value_counts())
print("\nFractie positief:", df["p_np"].mean().round(3))

In [ ]:
print("Missende SMILES:", df["smiles"].isna().sum())
df = df.dropna(subset=["smiles"]).reset_index(drop=True)

In [ ]:
from sklearn.model_selection import train_test_split

smiles = df["smiles"].tolist()
labels = df["p_np"].astype(int).tolist()

# Eerst test eraf halen (10%)
train_smiles, test_smiles, train_labels, test_labels = train_test_split(
    smiles, labels, test_size=0.1, random_state=42, stratify=labels
)
# Daarna val van train (10% van de rest)
train_smiles, val_smiles, train_labels, val_labels = train_test_split(
    train_smiles, train_labels, test_size=0.1, random_state=42, stratify=train_labels
)

print(f"Train: {len(train_smiles)} | Val: {len(val_smiles)} | Test: {len(test_smiles)}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "DeepChem/ChemBERTa-77M-MLM"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

In [ ]:
sample = "CC(=O)Oc1ccccc1C(=O)O"  # aspirine
tokens = tokenizer.tokenize(sample)
print("SMILES:", sample)
print("Tokens:", tokens)
print("Aantal tokens:", len(tokens))


In [ ]:
import torch
from torch.utils.data import Dataset

class SMILESDataset(Dataset):
    def __init__(self, smiles, labels, tokenizer, max_length=128):
        self.smiles = smiles
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.smiles[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = SMILESDataset(train_smiles, train_labels, tokenizer)
val_ds   = SMILESDataset(val_smiles,   val_labels,   tokenizer)
test_ds  = SMILESDataset(test_smiles,  test_labels,  tokenizer)

print("Aantal samples in train_ds:", len(train_ds))
print("Voorbeeld input_ids shape:", train_ds[0]["input_ids"].shape)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        "accuracy": accuracy_score(labels, preds),
        "roc_auc":  roc_auc_score(labels, probs),
    }

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./chemberta_bbbp",
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="roc_auc",
    greater_is_better=True,
    logging_steps=20,
    save_total_limit=2,
    report_to="none",
    fp16=True,            # mixed-precision: sneller op T4 GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
test_metrics = trainer.evaluate(test_ds)

print("=== Testresultaten ===")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k:25s}: {v:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Voorspellingen op de test-set
predictions = trainer.predict(test_ds)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix:")
print("                 voorspeld 0   voorspeld 1")
print(f"  werkelijk 0:    {cm[0,0]:5d}        {cm[0,1]:5d}")
print(f"  werkelijk 1:    {cm[1,0]:5d}        {cm[1,1]:5d}")

print("\nPer-klasse metrics:")
print(classification_report(y_true, y_pred, target_names=["geen BBB (0)", "wel BBB (1)"]))